# 04 — Online evaluation

Run the streaming engine via the **single protocol**. ARF + Mondrian-ACI learn online from the LabelBuffer; SimBroker walks first-touch over each 20-min decision bar; RiskEngine arbitrates every Action; MetricsBattery + ChartBattery + Report write `artifacts/runs/<run_id>/`.

There is no second way to do this — the protocol is the only entry. Notebook just demonstrates the in-process API; equivalent CLI is `wagie experiment run experiments/baseline.yaml`.

In [ ]:
from pathlib import Path
from wagie.experiments import ExperimentSpec, ExperimentProtocol

raw = Path("data/synthetic/btcusdt_1m.parquet")
assert raw.is_file(), "run 01_data_download first"

spec = ExperimentSpec.model_validate({
    "name": "notebook_eval",
    "description": "online_eval recreated as a notebook over the wagie package",
    "seed": 42,
    "wagie": {
        "data": {"parquet_path": str(raw), "m_minutes": 20},
        "model": {
            # Drop the offline model in here once 03_offline_train has produced it:
            # "catboost_path": "artifacts/offline_model/model.cbm",
            # "selected_features_path": "artifacts/offline_model/selected_features.json",
            "catboost_path": None,
            "arf": {"n_models": 10, "lambda_value": 6.0, "seed": 42},
            "aci": {"alphas": [0.05, 0.10, 0.20], "gamma": 0.01, "q_init": 0.5},
        },
        "strategy": {"kind": "pure_conformal", "alpha": 0.10},
        "broker": {"inventory_cap": 5},
        "runtime": {"warmup_samples": 96},
    },
    "features": {"catalog": "default"},
    "charts": {"enable": True, "n_calibration_bins": 10},
    "report": {"enable": True},
    "artifacts": {"out_dir": "artifacts/runs"},
})
spec.hash()

In [ ]:
result = ExperimentProtocol().run(spec)
print(result.headline)
print("out_dir:", result.out_dir)

Inspect the metrics — calibration leads, then trading, then conformal coverage:

In [ ]:
import json
metrics = json.loads((result.out_dir / "metrics.json").read_text())
print(f"Brier:  {metrics['brier']:.5f}")
print(f"ECE:    {metrics['ece']:.5f}")
t = metrics['trading']
print(f"trades: {t['n_trades']}  Sharpe={t['sharpe']:+.3f}  PSR={t['probabilistic_sharpe']:.3f}")
print(f"hit_rate={t['hit_rate']:.3f}  max_dd={t['max_drawdown_log']:.4f}")

In [ ]:
# Inline the rendered report (Markdown).
from IPython.display import Markdown
Markdown((result.out_dir / "report.md").read_text())

## Cross-validation mode

Same protocol, same spec — just toggle `spec.cv`. The protocol delegates to `wagie.cv.cross_validation` under the hood and aggregates per-fold metrics into the same MetricsReport / Report shape, plus a per-fold Sharpe panel chart.

In [ ]:
from wagie.experiments import CVSpec
import dataclasses

spec_cv = spec.model_copy(deep=True)
spec_cv.name = "notebook_eval_cv"
spec_cv.cv = CVSpec(enabled=True, n_folds=4, n_test_folds=2, embargo_size=2, purged_size=1)

cv_result = ExperimentProtocol().run(spec_cv)
print(cv_result.headline)
print("out_dir:", cv_result.out_dir)

Per-fold table:

In [ ]:
import json, polars as pl
m = json.loads((cv_result.out_dir / "metrics.json").read_text())
print(f"PBO (Bailey-LdP): {m['pbo']:.3f}")
pl.DataFrame(m["per_fold"])

## Same flow from the shell

Save your spec to `experiments/notebook_eval.yaml` and run:

```bash
wagie experiment run  experiments/notebook_eval.yaml
wagie experiment list
wagie experiment show <run_id>
wagie cv             experiments/notebook_eval.yaml --n-folds 4 --n-test-folds 2 --embargo 2
```